# SASRec retrieval on Yambda-50M

Optional GPU experiment: a SASRec next-item model as an extra candidate source, scored on
exactly the same test requests as the rest of the system. The notebook only produces ranked
lists (`results/sasrec_test_lists.parquet`); metrics are computed by `recsys.train fit`, so
every method goes through one evaluation path.

Runs top to bottom on a free Colab or Kaggle T4 (6 epochs took about 20 minutes on an M4 with MPS). It also runs on Apple
silicon (MPS) or CPU, just slower. To bring the results back: copy `results/sasrec_*` into the
repo's `results/` and rerun `uv run python -m recsys.train fit`.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/egkiastas/realtime-recsys-platform"
ROOT = os.environ.get("RECSYS_ROOT", "realtime-recsys-platform")
if not os.path.exists(ROOT):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], check=True)
# same polars as the lock file, so the request sampling (seeded shuffle) is identical
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "polars==2.1.0", "huggingface-hub==2.4.0"], check=True)
sys.path.insert(0, os.path.abspath(ROOT))

In [ ]:
import json
import time

import numpy as np
import polars as pl
import torch
from huggingface_hub import hf_hub_download
from torch import nn

from recsys.config import load_config
from recsys.data import REPO, REVISION, load_events, make_requests

cfg = load_config(os.path.join(ROOT, "config.toml"))
if not (cfg.raw_dir / "flat/50m/multi_event.parquet").exists():
    hf_hub_download(REPO, "flat/50m/multi_event.parquet", repo_type="dataset", revision=REVISION, local_dir=cfg.raw_dir)
events = load_events(cfg)
requests, _ = make_requests(events, cfg.test_start, cfg.last_ts + 1, cfg)

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
MAXLEN, DIM, BLOCKS, HEADS, DROPOUT = 200, 64, 2, 2, 0.2
EPOCHS, BATCH, LR, N_NEG, TRAIN_DAYS = 6, 128, 1e-3, 1024, 120
print(device, len(requests), "test requests")

In [ ]:
listens = (
    events.filter(pl.col("event_type") == "listen")
    .select("uid", "timestamp", "item_id")
    .sort("uid", "timestamp", maintain_order=True)
)
train = listens.filter(pl.col("timestamp") < cfg.train_end)
vocab = np.sort(train.group_by("item_id").len().filter(pl.col("len") >= cfg.min_item_count)["item_id"].to_numpy())
PAD, UNK = 0, 1


def tokens(item_ids: np.ndarray) -> np.ndarray:
    pos = np.searchsorted(vocab, item_ids).clip(max=len(vocab) - 1)
    return np.where(vocab[pos] == item_ids, pos + 2, UNK)


# non-overlapping windows of MAXLEN+1 from the end of each user's recent training history
windows = []
recent = train.filter(pl.col("timestamp") >= cfg.train_end - TRAIN_DAYS * cfg.day)
for _, g in recent.group_by("uid", maintain_order=True):
    seq = tokens(g["item_id"].to_numpy())
    for end in range(len(seq), 1, -MAXLEN):
        w = seq[max(0, end - MAXLEN - 1) : end]
        windows.append(np.pad(w, (MAXLEN + 1 - len(w), 0)))
windows = torch.from_numpy(np.stack(windows)).long()
print(f"{len(vocab)} items in vocabulary, {len(windows)} training windows")

In [ ]:
class SASRec(nn.Module):
    def __init__(self, n_tokens: int):
        super().__init__()
        self.item = nn.Embedding(n_tokens, DIM, padding_idx=PAD)
        self.pos = nn.Embedding(MAXLEN, DIM)
        layer = nn.TransformerEncoderLayer(DIM, HEADS, DIM * 4, DROPOUT, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, BLOCKS)
        self.norm = nn.LayerNorm(DIM)
        self.drop = nn.Dropout(DROPOUT)
        # small init: N(0, 1) embeddings with tied output weights start with logits around +-40,
        # the softmax saturates and every user ends up with nearly the same representation
        nn.init.normal_(self.item.weight, std=0.02)
        nn.init.normal_(self.pos.weight, std=0.02)
        self.register_buffer("causal", torch.triu(torch.ones(MAXLEN, MAXLEN, dtype=torch.bool), 1))

    def forward(self, seq: torch.Tensor) -> torch.Tensor:
        x = self.item(seq) * DIM**0.5 + self.pos(torch.arange(seq.shape[1], device=seq.device))
        x = x * (seq != PAD).unsqueeze(-1)
        # No src_key_padding_mask: with one, PyTorch's inference fast path returns NaN for
        # left-padded sequences in eval mode. Padded positions are zeroed instead, as in the
        # original SASRec, and the causal mask does the rest.
        x = self.encoder(self.drop(x), mask=self.causal[: seq.shape[1], : seq.shape[1]])
        return self.norm(x)


model = SASRec(len(vocab) + 2).to(device)
i = torch.arange(MAXLEN, device=device)
recent_band = (i[None, :] <= i[:, None]) & (i[None, :] > i[:, None] - cfg.last_n)  # [target pos, input pos]
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
freq = torch.bincount(windows[:, 1:].flatten(), minlength=len(vocab) + 2).float()
freq[:2] = 0
q = (freq / freq.sum()).to(device)
log_q = torch.log(q + 1e-12)
log = []
for epoch in range(EPOCHS):
    model.train()
    t, losses = time.time(), []
    for batch in torch.randperm(len(windows)).split(BATCH):
        w = windows[batch].to(device)
        inp, tgt = w[:, :-1], w[:, 1:]
        h = model(inp)
        # Same rule as the labels and the two-tower: a target already among the previous
        # last_n items is a repeat, which the product never recommends, so it is not trained on.
        seen = (tgt.unsqueeze(2) == inp.unsqueeze(1)) & recent_band
        mask = (tgt > UNK) & ~seen.any(-1)
        h, tgt = h[mask], tgt[mask]
        # sampled softmax with negatives drawn by frequency and the logQ correction, so popular
        # items get pushed down as often as they are pushed up (uniform negatives collapse to
        # recommending the same few dozen hits)
        neg = torch.multinomial(q, N_NEG, replacement=True)
        pos_logit = (h * model.item(tgt)).sum(-1, keepdim=True) - log_q[tgt].unsqueeze(1)
        logits = torch.cat([pos_logit, h @ model.item(neg).T - log_q[neg]], dim=1)
        loss = nn.functional.cross_entropy(logits, torch.zeros(len(tgt), dtype=torch.long, device=device))
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    log.append({"epoch": epoch, "loss": float(np.mean(losses)), "seconds": round(time.time() - t)})
    print(log[-1])

In [ ]:
# Score every test request from the user's listens strictly before the request, the same
# information the online system has. Recently played items are dropped as everywhere else.
model.eval()
by_user = {uid: (g["timestamp"].to_numpy(), g["item_id"].to_numpy()) for (uid,), g in listens.group_by("uid")}
item_emb = model.item.weight[2:].detach()
out_ids, out_lists = [], []
reqs = requests.to_dicts()
with torch.no_grad():
    for start in range(0, len(reqs), 256):
        chunk = reqs[start : start + 256]
        seqs, recents = [], []
        for r in chunk:
            ts, items = by_user[r["uid"]]
            hist = items[: np.searchsorted(ts, r["request_ts"], side="right")]
            recents.append(set(hist[-cfg.last_n :].tolist()))
            seq = tokens(hist[-MAXLEN:])
            seqs.append(np.pad(seq, (MAXLEN - len(seq), 0)))
        h = model(torch.from_numpy(np.stack(seqs)).long().to(device))[:, -1]
        top = torch.topk(h @ item_emb.T, 300).indices.cpu().numpy()
        for r, cand, rec in zip(chunk, top, recents, strict=True):
            ids = [int(i) for i in vocab[cand] if int(i) not in rec][:200]
            out_ids.append(r["request_id"])
            out_lists.append(ids)

os.makedirs(os.path.join(ROOT, "results"), exist_ok=True)
pl.DataFrame(
    {"request_id": out_ids, "items": out_lists}, schema={"request_id": pl.UInt32, "items": pl.List(pl.Int64)}
).write_parquet(os.path.join(ROOT, "results", "sasrec_test_lists.parquet"))
with open(os.path.join(ROOT, "results", "sasrec_meta.json"), "w") as f:
    json.dump(
        {"device": str(device), "epochs": log, "maxlen": MAXLEN, "dim": DIM, "blocks": BLOCKS, "vocab": len(vocab)}, f, indent=2
    )
print("wrote", len(out_ids), "ranked lists")